# FieldAI: Receipt Information Extraction with Layout-Aware Transformers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Disha-01-alt/field-ai/blob/main/notebooks/FieldAI_Complete_Colab_Benchmark.ipynb)

## Project Overview & Research Angle
This notebook trains and benchmarks three model paradigms on the **ICDAR 2019 SROIE** dataset:
1. **Baseline 1:** Deterministic OCR + Regular Expression Extractor
2. **Baseline 2:** `bert-base-uncased` Token Classification (1D Text Only)
3. **Multimodal Model:** `microsoft/layoutlm-base-uncased` Token Classification (Text + 2D Spatial Layout)

> **Core Research Question:**  
> $$\Delta F1_{\text{field}} = F1_{\text{LayoutLM}}(\text{field}) - F1_{\text{BERT}}(\text{field})$$
> *How much does 2D spatial arrangement improve extraction for Company, Date, and Total over text alone?*

---  
### Step 1: Verify Hardware Acceleration (GPU)

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. For fast training, enable GPU via Runtime > Change runtime type > T4 GPU.")

---  
### Step 2: Clone Repository & Install Dependencies

In [ ]:
# If running inside Colab, clone the project repository
import os
from pathlib import Path

if not Path("src").exists():
    !git clone https://github.com/Disha-01-alt/field-ai.git
    %cd field-ai

!pip install -q -r requirements.txt
print("\nDependencies installed and workspace configured!")

---  
### Step 3: Inspect the Preprocessed SROIE Dataset
The dataset contains **500 training receipts** and **126 test receipts** with coordinates normalized to $[0, 1000]$ and BIO entity labels.

In [ ]:
import json

with open("data/processed/train.json", "r") as f:
    train_data = json.load(f)
with open("data/processed/test.json", "r") as f:
    test_data = json.load(f)

print(f"Training Receipts: {len(train_data)}")
print(f"Testing Receipts: {len(test_data)}")

sample = test_data[0]
print("\n--- Sample Test Document ---")
print(f"ID: {sample['id']}")
print(f"Target Entities: {sample['entities']}")
print("First 5 words with normalized 2D bboxes:")
for w, b, tag in zip(sample['words'][:5], sample['bboxes'][:5], sample['bio_tags'][:5]):
    print(f"  {w:<15} | BBox: {str(b):<20} | Tag: {tag}")

---  
### Step 4: Run Baseline 1 (OCR + Regex Rules)
Evaluates deterministic pattern heuristics for date, proximity search for total, and line-position for company on the 126 test receipts.

In [ ]:
!PYTHONPATH=. python src/baseline_regex.py

---  
### Step 5: Fine-Tune Model 2 — BERT-base (Text-Only Transformer)
Fine-tunes `bert-base-uncased` for token classification using 1D sequential WordPiece text tokens (without spatial coordinates).

In [ ]:
from src.train_bert import train_bert

# 5 epochs on 500 samples takes ~3 minutes on a Colab T4 GPU
bert_results = train_bert(
    train_file="data/processed/train.json",
    test_file="data/processed/test.json",
    output_path="results/bert_results.json",
    epochs=5,
    batch_size=8,
    lr=3e-5,
)

---  
### Step 6: Fine-Tune Model 3 — LayoutLM-base (Multimodal: Text + 2D Layout)
Fine-tunes `microsoft/layoutlm-base-uncased` using WordPiece text tokens and normalized 2D spatial bounding boxes $[x_0, y_0, x_1, y_1]$.

In [ ]:
from src.train_layoutlm import train_layoutlm

# 5 epochs on 500 samples takes ~4 minutes on a Colab T4 GPU
layoutlm_results = train_layoutlm(
    train_file="data/processed/train.json",
    test_file="data/processed/test.json",
    output_path="results/layoutlm_results.json",
    bert_results_path="results/bert_results.json",
    epochs=5,
    batch_size=8,
    lr=5e-5,
)

---  
### Step 7: Final Benchmark Table & Field-Wise $\Delta F1$ Visualizations

In [ ]:
import json
from src.metrics import format_results_markdown_table, compute_delta_f1
from src.visualize import generate_svg_chart, generate_matplotlib_plot

with open("results/regex_baseline.json") as f:
    regex_m = json.load(f)
with open("results/bert_results.json") as f:
    bert_m = json.load(f)
with open("results/layoutlm_results.json") as f:
    layout_m = json.load(f)

all_models = {
    "OCR + Regex Baseline": regex_m,
    "BERT-base (Text Only)": bert_m,
    "LayoutLM-base (Multimodal)": layout_m,
}

# Print Markdown Summary Table
table_md = format_results_markdown_table(all_models)
print("### Final Benchmark Results Table:\n")
print(table_md)

# Compute and Print Empirical Delta
deltas = compute_delta_f1(bert_m["metrics"], layout_m["metrics"])
print("\n### Empirical ΔF1 = F1(LayoutLM) - F1(BERT):")
for field, val in deltas.items():
    print(f"  {field.upper():<12}: {val * 100:+.2f}%")

# Generate Visualization Plots
benchmark_chart_data = {
    model_name: {k: v["f1"] for k, v in data["metrics"].items() if "f1" in v}
    for model_name, data in all_models.items()
}
generate_svg_chart(benchmark_chart_data, "results/f1_comparison.svg")
generate_matplotlib_plot(benchmark_chart_data, "results/f1_comparison.png")

---  
### Step 8: View the Generated Benchmark Plot Inline

In [ ]:
from IPython.display import Image, display
if os.path.exists("results/f1_comparison.png"):
    display(Image(filename="results/f1_comparison.png"))
else:
    print("Chart saved as results/f1_comparison.svg")